In [2]:
!python tutorial_train.py


/root/miniconda3/envs/control/lib/python3.8/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: libtorch_cuda_cu.so: cannot open shared object file: No such file or directory
  warn(f"Failed to load image Python extension: {e}")
logging improved.
/root/miniconda3/envs/control/lib/python3.8/site-packages/pytorch_lightning/plugins/training_type/ddp.py:68: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import DistributedOptimizer
/root/miniconda3/envs/control/lib/python3.8/site-packages/pytorch_lightning/core/lightning.py:2058: DeprecationWarning: `torch.distributed._sharded_tensor` will be deprecated, use `torch.distributed._shard.sharded_tensor` instead
  from torch.distributed._sharded_tensor import pre_load_state_dict_hook, state_dict_hook
ControlLDM: Running in eps-prediction 

In [ ]:
# 🔧 1. Import thư viện
import os
from PIL import Image
import numpy as np
import torch
import clip
from torchvision import transforms
from pytorch_fid import fid_score
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# ✅ 2. Thiết lập đường dẫn
base_dir = "./logs/image_log/train"
conditioning_dir = os.path.join(base_dir, "conditioning")
reconstruction_dir = os.path.join(base_dir, "reconstruction")
samples_dir = os.path.join(base_dir, "samples_cfg_scale_9.00")

# ✅ 3. Load CLIP model
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, preprocess = clip.load("ViT-B/32", device=device)

# ✅ 4. Load ảnh và tiền xử lý
def load_image(path):
    image = Image.open(path).convert("RGB")
    return preprocess(image).unsqueeze(0).to(device)

def load_images_from_folder(folder):
    files = sorted([f for f in os.listdir(folder) if f.endswith(".png")])
    paths = [os.path.join(folder, f) for f in files]
    images = [load_image(p) for p in paths]
    return files, images

conditioning_files, conditioning_images = load_images_from_folder(conditioning_dir)
samples_files, samples_images = load_images_from_folder(samples_dir)
recon_files, _ = load_images_from_folder(reconstruction_dir)  # only for filenames

# ✅ 5. Tính CLIP Score giữa conditioning ↔ samples
clip_scores = []
for cond_img, gen_img in zip(conditioning_images, samples_images):
    cond_feat = clip_model.encode_image(cond_img).detach().cpu().numpy()
    gen_feat = clip_model.encode_image(gen_img).detach().cpu().numpy()
    score = cosine_similarity(cond_feat, gen_feat)[0][0]
    clip_scores.append(score)

# ✅ 6. Tính FID giữa reconstruction ↔ samples
fid_value = fid_score.calculate_fid_given_paths(
    [reconstruction_dir, samples_dir],
    batch_size=8,
    device=device,
    dims=2048
)

# ✅ 7. Hiển thị kết quả
df = pd.DataFrame({
    "conditioning_file": conditioning_files,
    "sample_file": samples_files,
    "clip_score": clip_scores
})

print("🎯 FID Score (reconstruction vs samples):", fid_value)
df.head(10)


Downloading: "https://github.com/mseitzer/pytorch-fid/releases/download/fid_weights/pt_inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/pt_inception-2015-12-05-6726825d.pth
100%|██████████| 91.2M/91.2M [00:02<00:00, 37.6MB/s]
100%|██████████| 22/22 [00:01<00:00, 13.06it/s]


🎯 FID Score (reconstruction vs samples): 87.03446365100822


,conditioning_file,sample_file,clip_score
0,conditioning_gs-000000_e-000000_b-000000_i-00_...,samples_cfg_scale_9.00_gs-000000_e-000000_b-00...,0.520878
1,conditioning_gs-000000_e-000000_b-000000_i-01_...,samples_cfg_scale_9.00_gs-000000_e-000000_b-00...,0.590421
2,conditioning_gs-000000_e-000000_b-000000_i-02_...,samples_cfg_scale_9.00_gs-000000_e-000000_b-00...,0.605882
3,conditioning_gs-000000_e-000000_b-000000_i-03_...,samples_cfg_scale_9.00_gs-000000_e-000000_b-00...,0.635238
4,conditioning_gs-000300_e-000000_b-000300_i-00_...,samples_cfg_scale_9.00_gs-000300_e-000000_b-00...,0.661827
5,conditioning_gs-000300_e-000000_b-000300_i-01_...,samples_cfg_scale_9.00_gs-000300_e-000000_b-00...,0.574132
6,conditioning_gs-000300_e-000000_b-000300_i-02_...,samples_cfg_scale_9.00_gs-000300_e-000000_b-00...,0.609766
7,conditioning_gs-000300_e-000000_b-000300_i-03_...,samples_cfg_scale_9.00_gs-000300_e-000000_b-00...,0.646119
8,conditioning_gs-000600_e-000000_b-000600_i-00_...,samples_cfg_scale_9.00_gs-000600_e-000000_b-00...,0.627663
9,conditioning_gs-000600_e-000000_b-000600_i-01_...,samples_cfg_scale_9.00_gs-000600_e-000000_b-00...,0.728692


In [46]:
import os
import json
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity

# 📁 Đường dẫn
control_dir = "./logs/image_log/train/control"
source_dir = "./training/fill50k/source"
target_dir = "./training/fill50k/target"
recon_dir = "./logs/image_log/train/reconstruction"
sample_dir = "./logs/image_log/train/samples"
prompt_path = "./training/fill50k/prompt.json"

# 📌 Load prompt.json
with open(prompt_path, "r") as f:
    prompt_data = [json.loads(line.strip()) for line in f]

# 🔎 Lập chỉ mục theo prompt
prompt_to_data = {}
for item in prompt_data:
    prompt = item["prompt"]
    prompt_to_data.setdefault(prompt, []).append(item)

# 🖼️ Hàm load ảnh an toàn
def load_image(path):
    if not os.path.exists(path):
        print(f"⚠️ File không tồn tại: {path}")
        return None
    img = cv2.imread(path)
    if img is None:
        print(f"⚠️ Không thể đọc ảnh: {path}")
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (64, 64))
    img = img.astype(np.float32) / 255.0
    return img.flatten()

# 🔄 Quét toàn bộ ảnh trong control/
mapping = []
for filename in tqdm(os.listdir(control_dir)):
    if not filename.endswith(".png"):
        continue

    try:
        filename_core = filename.replace("control_", "").replace(".png", "")  # bỏ tiền tố và đuôi
        split_idx = filename_core.find("_i-")
        if split_idx == -1:
            print(f"❌ Không tìm thấy _i- trong tên file: {filename}")
            continue
        prefix = filename_core[:split_idx]  # vd: gs-002100_e-000000_b-002100
        prompt_sanitized = filename_core[split_idx + 4:]  # bỏ luôn 'i-XX_' => lấy phần prompt
        i_number = filename_core[split_idx+3:split_idx+5]  # lấy chỉ số i (00, 01, 03, ...)
        prompt = " ".join(prompt_sanitized.split("_"))

    except Exception as e:
        print(f"❌ Lỗi khi parse {filename}: {e}")
        continue

    candidates = prompt_to_data.get(prompt, [])
    if not candidates:
        continue

    control_path = os.path.join(control_dir, filename)
    control_img = load_image(control_path)
    if control_img is None:
        continue

    best_score = -1
    best_source, best_target = None, None
    for item in candidates:
        source_path = os.path.join("./training/fill50k", item["source"])
        source_img = load_image(source_path)
        if source_img is None:
            continue
        score = cosine_similarity([control_img], [source_img])[0][0]
        if score > best_score:
            best_score = score
            best_source = item["source"]
            best_target = item["target"]

    if best_source and best_target:
        recon_filename = f"reconstruction_{prefix}_i-{i_number}_{prompt_sanitized}.png"
        sample_filename = f"samples_cfg_scale_9.00_{prefix}_i-{i_number}_{prompt_sanitized}.png"
        mapping.append({
            "control": filename,
            "prompt": prompt,
            "source": best_source,
            "target": best_target,
            "reconstruction": recon_filename,
            "samples": sample_filename,
            "similarity": round(best_score, 4)
        })

# 💾 Lưu file CSV
df = pd.DataFrame(mapping)
df.to_csv("./logs/image_log/train/mapping_matched_3.csv", index=False)
print("✅ mapping_matched.csv đã được cập nhật đúng định dạng!")


100%|██████████| 176/176 [00:00<00:00, 326780.66it/s]

✅ mapping_matched.csv đã được cập nhật đúng định dạng!


In [50]:
import os
import shutil
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity

import clip
import torch
from torchvision import transforms
from pytorch_fid import fid_score

# ===== Config paths =====
base_path = "./training/fill50k"
recon_dir = "./logs/image_log/train/reconstruction"
sample_dir = "./logs/image_log/train/samples_cfg_scale_9.00"
target_dir = os.path.join(base_path, "target")
mapping_file = "./logs/image_log/train/mapping_matched.csv"

# ===== Load CLIP model =====
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, preprocess = clip.load("ViT-B/32", device=device)

# ===== Helper: Load & preprocess image =====
def load_and_preprocess(path):
    image = Image.open(path).convert("RGB")
    return preprocess(image).unsqueeze(0).to(device)

# ===== Prepare folders for FID =====
fid_temp_target = "./fid_temp/target"
fid_temp_recon = "./fid_temp/reconstruction"
fid_temp_sample = "./fid_temp/samples"
for folder in [fid_temp_target, fid_temp_recon, fid_temp_sample]:
    shutil.rmtree(folder, ignore_errors=True)
    os.makedirs(folder, exist_ok=True)

# ===== Load mapping =====
df = pd.read_csv(mapping_file)
clip_results = []

valid_indices = []  # Chỉ giữ index hợp lệ để dùng cho FID

# ===== Main loop =====
for idx, row in tqdm(df.iterrows(), total=len(df)):
    try:
        prompt = row['prompt']
        target_path = os.path.join(base_path, row['target'])
        recon_path = os.path.join(recon_dir, row['reconstruction'])
        sample_path = os.path.join(sample_dir, row['samples'])

        # Check file tồn tại
        if not (os.path.exists(target_path) and os.path.exists(recon_path) and os.path.exists(sample_path)):
            print(f"[⚠️ Bỏ qua index {idx}]: thiếu file.")
            continue

        # Copy ảnh cho FID
        shutil.copy(target_path, os.path.join(fid_temp_target, f"{idx}.png"))
        shutil.copy(recon_path, os.path.join(fid_temp_recon, f"{idx}.png"))
        shutil.copy(sample_path, os.path.join(fid_temp_sample, f"{idx}.png"))
        valid_indices.append(idx)

        # CLIP encode (disable gradient computation)
        with torch.no_grad():
            target_feat = clip_model.encode_image(load_and_preprocess(target_path))
            recon_feat = clip_model.encode_image(load_and_preprocess(recon_path))
            sample_feat = clip_model.encode_image(load_and_preprocess(sample_path))

        # Convert to numpy safely
        recon_score = cosine_similarity(
            target_feat.detach().cpu().numpy(),
            recon_feat.detach().cpu().numpy()
        )[0][0]
        sample_score = cosine_similarity(
            target_feat.detach().cpu().numpy(),
            sample_feat.detach().cpu().numpy()
        )[0][0]

        clip_results.append({
            "index": idx,
            "prompt": prompt,
            "clip_reconstruction": recon_score,
            "clip_sample": sample_score
        })

    except Exception as e:
        print(f"[❌ Lỗi tại index {idx}]: {e}")

# ===== Save CLIP Scores =====
clip_df = pd.DataFrame(clip_results)
clip_df.to_csv("clip_scores.csv", index=False)

# ===== Calculate FID only if có ảnh đủ
if len(valid_indices) > 0:
    fid_recon = fid_score.calculate_fid_given_paths([fid_temp_target, fid_temp_recon], 50, device, 2048)
    fid_sample = fid_score.calculate_fid_given_paths([fid_temp_target, fid_temp_sample], 50, device, 2048)
else:
    fid_recon, fid_sample = -1, -1  # Đánh dấu lỗi

# ===== Save Summary =====
summary = pd.DataFrame([{
    "FID_Reconstruction": fid_recon,
    "FID_Sample": fid_sample,
    "CLIP_Reconstruction_Mean": clip_df["clip_reconstruction"].mean() if not clip_df.empty else -1,
    "CLIP_Sample_Mean": clip_df["clip_sample"].mean() if not clip_df.empty else -1
}])
summary.to_csv("score_summary.csv", index=False)

# ===== Print Output to Console =====
print("\n✅ Đã tính xong FID và CLIP. Kết quả:")

# Hiển thị bảng CLIP scores
print("\n📄 CLIP Scores (10 dòng đầu):")
print(clip_df.head(10).to_string(index=False))

# Hiển thị bảng tổng kết
print("\n📊 Score Summary:")
print(summary.to_string(index=False))


100%|██████████| 44/44 [00:03<00:00, 11.84it/s]


100%|██████████| 1/1 [00:00<00:00,  1.66it/s]


100%|██████████| 1/1 [00:00<00:00,  1.45it/s]


100%|██████████| 1/1 [00:00<00:00,  1.67it/s]


100%|██████████| 1/1 [00:00<00:00,  1.38it/s]



✅ Đã tính xong FID và CLIP. Kết quả:

📄 CLIP Scores (10 dòng đầu):
 index                                                  prompt  clip_reconstruction  clip_sample
     0            pale green circle with dark green background             0.906489     0.834099
     1                  lime circle with light gray background             0.971812     0.904897
     2                  khaki circle with corn silk background             0.983703     0.915068
     3            sandy brown circle with chocolate background             0.982398     0.870790
     4                   thistle circle with yellow background             0.967299     0.787970
     5 light yellow circle with medium spring green background             0.985025     0.861771
     6               dark blue circle with lavender background             0.992205     0.809523
     7                     crimson circle with aqua background             0.987365     0.861689
     8          floral white circle with cadet blue backgro